In [2]:
!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/nyc_tripdata_2024_sample_4M.csv
from pyspark.sql import SparkSession
spark = (
SparkSession.builder
.appName("ExerciciosPySpark")
.master("local[*]")
.getOrCreate()
)
df = spark.read.csv("nyc_tripdata_2024_sample_4M.csv", header=True,
inferSchema=True)

# Questão 1

In [3]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [5]:
df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-10-01 00:59:55|  2024-10-01 01:02:24|              1|          0.5|         1|                 N|         230|         161|           1|        5.1|  3.5|    0.5|       2.

In [6]:
df.count()

4118743

# Questão 2

In [7]:
df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "payment_type"
).show(5)

+--------+--------------------+-------------+-----------+------------+
|VendorID|tpep_pickup_datetime|trip_distance|fare_amount|payment_type|
+--------+--------------------+-------------+-----------+------------+
|       1| 2024-10-01 00:59:55|          0.5|        5.1|           1|
|       1| 2024-10-01 00:08:59|         20.6|       76.5|           2|
|       2| 2024-10-01 00:18:38|         7.42|       33.1|           4|
|       2| 2024-10-01 00:20:06|        19.96|       70.0|           1|
|       1| 2024-10-01 00:09:02|          2.6|       15.6|           1|
+--------+--------------------+-------------+-----------+------------+
only showing top 5 rows


# Questão 3

In [8]:
filtro = df.filter(
    (df.trip_distance > 5) &
    (df.passenger_count >= 3)
)

filtro.count()

50665

# Questão 4

Quando utilizamos inferSchema=True, o Spark tenta automaticamente identificar o tipo de cada coluna do CSV. As vantagens são: um código menor e mais praticidade. As desvantagens são: o Sparky pode inferir tipos incorretamente se houver valores inconsistentes. No schema manual há maior controle, porém exige mais trabalho e é necessário conhecer previamente a estrutura dos dados.

# Questão 5

In [9]:
from pyspark.sql.functions import count, sum

In [10]:
resultado = (
    df.groupBy("payment_type")
      .agg(
          count("*").alias("quantidade_corridas"),
          sum("total_amount").alias("receita_total")
      )
      .orderBy("receita_total", ascending=False)
)

resultado.show()

+------------+-------------------+--------------------+
|payment_type|quantidade_corridas|       receita_total|
+------------+-------------------+--------------------+
|           1|            3045849| 9.116799616010016E7|
|           2|             553536|1.2987084559999354E7|
|           0|             410746|1.0123049400000528E7|
|           3|              29100|  220775.24999999974|
|           4|              79511|  133192.01999999984|
|           5|                  1|                62.0|
+------------+-------------------+--------------------+



# Questão 6

In [11]:
from pyspark.sql.functions import hour, avg

In [12]:
df_hora = df.withColumn(
    "hora_embarque",
    hour("tpep_pickup_datetime")
)

In [13]:
resultado = (
    df_hora.groupBy("hora_embarque")
    .agg(
        avg("fare_amount").alias("tarifa_media"),
        avg("trip_distance").alias("distancia_media")
    )
    .orderBy("hora_embarque")
)

resultado.show(24)

+-------------+------------------+------------------+
|hora_embarque|      tarifa_media|   distancia_media|
+-------------+------------------+------------------+
|            0| 19.72867660335703| 5.130178643081671|
|            1| 17.54847894641617| 3.739999483030465|
|            2|16.426538461538446| 4.542445678033303|
|            3| 17.24064640950263| 3.401675265462839|
|            4| 22.33585451861572|11.412191651631977|
|            5|26.226564065583663| 23.33988120540463|
|            6|21.931859821807333|14.540589393296582|
|            7| 19.33026953083623|11.087329050022918|
|            8|18.511148615351107| 8.533842520592145|
|            9|18.400291333656767| 5.604490502277248|
|           10| 18.56454835768904|4.5114450807098905|
|           11|18.851552824117647| 4.075729557436569|
|           12|19.207714073999153| 4.468694683646846|
|           13| 19.95819012628161| 5.262006204074442|
|           14|20.604332332373676| 4.622973056355365|
|           15| 20.757107834

# Questão 7

No Spark existem: transformações e ações. Transformações são processos que criam um novo DataFrame a partir de outro, como select() ; filter() ; groupBy();
já as ações, executam o processamento dos dados e retornam um resultado, como show() ou count().

O Spark utiliza o conceito de lazy evaluation, ou seja, ele registra as transformações que devem ser realizadas e espera até que uma ação seja chamada para executar tudo que está pendente. A vantagem prática é que assim é possível otimizar o plano de execução, eliminando etapas desnecessárias e reduzindo o volume de dados processados.

# Questão 8

In [15]:
from pyspark.sql.functions import col

In [16]:
resultado = (
    df.filter(col("total_amount") > 0)
    .withColumn(
        "percentual_gorjeta",
        (col("tip_amount") / col("total_amount")) * 100
    )
    .select(
        "VendorID",
        "total_amount",
        "tip_amount",
        "percentual_gorjeta"
    )
    .orderBy("percentual_gorjeta", ascending=False)
)

resultado.show(10)

+--------+------------+----------+------------------+
|VendorID|total_amount|tip_amount|percentual_gorjeta|
+--------+------------+----------+------------------+
|       2|        1.63|      5.27| 323.3128834355828|
|       2|        2.07|      3.68| 177.7777777777778|
|       2|         1.6|      2.82|176.24999999999997|
|       2|        2.33|      3.72|159.65665236051504|
|       2|        2.54|      3.76|148.03149606299212|
|       2|        3.76|      3.96|105.31914893617022|
|       2|        39.7|      40.0|100.75566750629723|
|       2|        0.08|      0.08|             100.0|
|       1|       197.0|     196.0| 99.49238578680203|
|       1|       150.0|     149.0| 99.33333333333333|
+--------+------------+----------+------------------+
only showing top 10 rows


# Questão 9

In [17]:
!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/taxi_zone_lookup.csv

zonas = spark.read.csv(
    "taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)

zonas.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [18]:
join_df = df.join(
    zonas,
    df.PULocationID == zonas.LocationID,
    "inner"
)

In [19]:
resultado = (
    join_df.groupBy("Borough")
    .count()
    .orderBy("count", ascending=False)
)

resultado.show()

+-------------+-------+
|      Borough|  count|
+-------------+-------+
|    Manhattan|3641752|
|       Queens| 388736|
|     Brooklyn|  60200|
|        Bronx|  12702|
|      Unknown|  12172|
|          N/A|   2421|
|          EWR|    565|
|Staten Island|    195|
+-------------+-------+



# Questão 10

O comando count() da Questão 1 apenas percorre o arquivo para contabilizar quantas linhas existem. Já o groupBy() da Questão 5 exige que registros pertencentes ao mesmo grupo estejam juntos para que os cálculos sejam realizados.

No processo de shuffle, os dados são distribuídos para diferentes partições e nós do cluster. Essa movimentação pela rede e a reorganização das partições tornam a operação mais custosa.

Por isso, operações de agrupamento geralmente demoram mais do que operações simples de seleção ou filtragem. Existe um custo adicional em reorganizar os dados para formar os grupos antes de realizar os cálculos agregados.